# THUOC - Colab End-to-End Notebook

Notebook nay duoc tao de chay ra day du ket qua cua do an THUOC trong Colab:
- Checkpoint: `*_epillid_best.pt`
- Metrics/history/curves
- `models/evaluation_summary.csv`, `models/evaluation_comparison.png`
- `models/reports/latest/*` (CSV, JSON, confusion matrix, tuning summary)

Ban chi can chay lan luot tu Cell 1 den het.

In [ ]:
# Cell 1 - Cau hinh
REPO_URL = "https://github.com/<your-user>/<your-repo>.git"  # TODO: sua lai repo cua ban
REPO_BRANCH = "main"
PROJECT_NAME = "THUOC"

# Neu data da nam trong repo thi de False
USE_DRIVE_DATA = True
DRIVE_DATA_ROOT = "/content/drive/MyDrive/THUOC_DATA"  # thu muc chua data_aligned hoac data

# Uu tien data_aligned, neu khong co se fallback data
PREFERRED_DATA_DIR = "data_aligned"

# Chon thiet bi
DEVICE = "cuda"

print("Config loaded")

In [ ]:
# Cell 2 - Mount Google Drive (neu can)
if USE_DRIVE_DATA:
    from google.colab import drive
    drive.mount('/content/drive')
    print('Drive mounted')
else:
    print('Skip Drive mount')

In [ ]:
# Cell 3 - Clone / cap nhat repo
import os
import subprocess
from pathlib import Path

ROOT = Path('/content')
PROJECT_DIR = ROOT / PROJECT_NAME

if PROJECT_DIR.exists():
    print(f'Project exists: {PROJECT_DIR}')
    subprocess.run(['git', '-C', str(PROJECT_DIR), 'fetch', '--all'], check=True)
    subprocess.run(['git', '-C', str(PROJECT_DIR), 'checkout', REPO_BRANCH], check=True)
    subprocess.run(['git', '-C', str(PROJECT_DIR), 'pull'], check=True)
else:
    if '<your-user>' in REPO_URL or '<your-repo>' in REPO_URL:
        raise ValueError('Ban can sua REPO_URL truoc khi chay Cell nay')
    subprocess.run(['git', 'clone', '--branch', REPO_BRANCH, REPO_URL, str(PROJECT_DIR)], check=True)

os.chdir(PROJECT_DIR)
print('Working dir:', os.getcwd())

In [ ]:
# Cell 4 - Cai dependencies
import sys
import torch

subprocess.run([sys.executable, '-m', 'pip', 'install', '--upgrade', 'pip'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', 'requirements.txt'], check=True)

print('Python:', sys.version.split()[0])
print('Torch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if DEVICE == 'cuda' and not torch.cuda.is_available():
    print('[WARN] CUDA khong available, script se tu fallback CPU theo logic project')

In [ ]:
# Cell 5 - Chuan bi du lieu
import shutil

def has_valid_splits(root: Path) -> bool:
    return (root / 'train').exists() and (root / 'val').exists() and (root / 'test').exists()

if USE_DRIVE_DATA:
    drive_root = Path(DRIVE_DATA_ROOT)
    if not drive_root.exists():
        raise FileNotFoundError(f'DRIVE_DATA_ROOT not found: {drive_root}')

    for name in ['data_aligned', 'data']:
        src = drive_root / name
        dst = PROJECT_DIR / name
        if src.exists() and not dst.exists():
            print(f'Copy {src} -> {dst}')
            shutil.copytree(src, dst)

data_dir = PROJECT_DIR / PREFERRED_DATA_DIR
if not has_valid_splits(data_dir):
    data_dir = PROJECT_DIR / 'data'

if not has_valid_splits(data_dir):
    raise FileNotFoundError('Khong tim thay data hop le. Can co train/val/test trong data_aligned hoac data')

print('Using data dir:', data_dir)

In [ ]:
# Cell 6 - Train + evaluate theo cau hinh hien tai (run_all.py)
cmd = [
    sys.executable,
    'run_all.py',
    '--data-dir', str(data_dir),
    '--device', DEVICE,
    '--output-dir', 'models'
]
print('Running:', ' '.join(cmd))
subprocess.run(cmd, check=True)

In [ ]:
# Cell 7 - Tao day du reports/latest tu checkpoint da train
import csv
import json
from dataclasses import asdict

import torch
from torch.utils.data import DataLoader

from src.features import PillImageDataset, build_transforms
from src.pipeline import (
    _evaluate_single_model,
    _evaluate_ensemble,
    _plot_confusion_matrix,
    _plot_score_comparison,
    _write_summary_csv,
)

models_dir = PROJECT_DIR / 'models'
report_dir = models_dir / 'reports' / 'latest'
report_dir.mkdir(parents=True, exist_ok=True)

dataset = PillImageDataset(str(data_dir / 'test'), transform=build_transforms(train=False))
loader = DataLoader(dataset, batch_size=16, shuffle=False, num_workers=0)
device = torch.device('cuda' if DEVICE == 'cuda' and torch.cuda.is_available() else 'cpu')

model_names = ['resnet50', 'efficientnet_b0', 'vit_b_16']
rows = []
pred_cache = {}

for model_name in model_names:
    ckpt = models_dir / f'{model_name}_epillid_best.pt'
    if not ckpt.exists():
        print(f'[WARN] missing checkpoint: {ckpt}')
        continue

    row, y_true, y_pred = _evaluate_single_model(
        model_name=model_name,
        checkpoint_path=ckpt,
        loader=loader,
        dataset_class_to_idx=dataset.class_to_idx,
        device=device,
    )
    rows.append(row)
    pred_cache[model_name] = (y_true, y_pred)

if not rows:
    raise RuntimeError('Khong co checkpoint hop le de tao report')

ensemble_row, y_true_ens, y_pred_ens = _evaluate_ensemble(
    model_names=[r.model for r in rows],
    models_dir=models_dir,
    loader=loader,
    dataset_class_to_idx=dataset.class_to_idx,
    device=device,
)
rows.append(ensemble_row)
pred_cache['ensemble_weighted'] = (y_true_ens, y_pred_ens)

# summary files
_write_summary_csv(rows, report_dir / 'evaluation_summary.csv')
with (report_dir / 'evaluation_summary.json').open('w', encoding='utf-8') as f:
    json.dump([asdict(r) for r in rows], f, indent=2, ensure_ascii=False)

_plot_score_comparison(rows, report_dir / 'evaluation_comparison.png')

# confusion matrix for each single model
for r in rows:
    name = r.model
    if name == 'ensemble_weighted':
        continue
    y_true, y_pred = pred_cache[name]
    _plot_confusion_matrix(
        y_true,
        y_pred,
        output_path=report_dir / f'confusion_matrix_{name}.png',
        title=f'Confusion Matrix - {name}',
    )

# confusion matrix ensemble
_plot_confusion_matrix(
    y_true_ens,
    y_pred_ens,
    output_path=report_dir / 'confusion_matrix_ensemble_weighted.png',
    title='Confusion Matrix - ensemble_weighted',
)

# tuning summary (lightweight)
tuning_summary = {
    'generated_by': 'THUOC_Colab_Train_Evaluate.ipynb',
    'data_dir': str(data_dir),
    'models_dir': str(models_dir),
    'models_evaluated': [r.model for r in rows],
}
with (report_dir / 'tuning_summary.json').open('w', encoding='utf-8') as f:
    json.dump(tuning_summary, f, indent=2, ensure_ascii=False)

print('Saved report dir:', report_dir)

In [ ]:
# Cell 8 - Kiem tra nhanh output
import pandas as pd

summary_top = PROJECT_DIR / 'models' / 'evaluation_summary.csv'
summary_report = PROJECT_DIR / 'models' / 'reports' / 'latest' / 'evaluation_summary.csv'

print('Top-level summary:', summary_top)
print('Report summary   :', summary_report)

if summary_top.exists():
    display(pd.read_csv(summary_top))

if summary_report.exists():
    display(pd.read_csv(summary_report))

In [ ]:
# Cell 9 - Dong goi ket qua de tai xuong / luu Drive
import shutil

zip_base = '/content/THUOC_models_artifacts'
zip_path = shutil.make_archive(zip_base, 'zip', root_dir=str(PROJECT_DIR / 'models'))
print('Created zip:', zip_path)

if USE_DRIVE_DATA:
    target = Path('/content/drive/MyDrive/THUOC_models_artifacts.zip')
    shutil.copy2(zip_path, target)
    print('Copied to:', target)